In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df = pd.read_csv("heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [7]:
df_original = pd.read_csv("heart.csv")
df["HeartDisease"] = df_original["HeartDisease"].astype(int)

df["HeartDisease"].unique()

array([0, 1])

In [8]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=["object"]).columns

print("Numéricas:", list(num_cols))
print("Categóricas:", list(cat_cols))


Numéricas: ['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak', 'HeartDisease']
Categóricas: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']


In [9]:
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

if len(cat_cols) > 0:
    df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

In [10]:
if len(cat_cols) > 0:
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

df.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,0,True,True,False,False,True,False,False,False,True
1,49,160,180,0,156,1.0,1,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,0,True,True,False,False,False,True,False,False,True
3,48,138,214,0,108,1.5,1,False,False,False,False,True,False,True,True,False
4,54,150,195,0,122,0.0,0,True,False,True,False,True,False,False,False,True


In [11]:
y = df["HeartDisease"]
X = df.drop("HeartDisease", axis=1)

In [12]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns

scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

X.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,-1.433140,0.410909,0.825070,-0.551341,1.382928,-0.832432,True,True,False,False,True,False,False,False,True
1,-0.478484,1.491752,-0.171961,-0.551341,0.754157,0.105664,False,False,True,False,True,False,False,True,False
2,-1.751359,-0.129513,0.770188,-0.551341,-1.525138,-0.832432,True,True,False,False,False,True,False,False,True
3,-0.584556,0.302825,0.139040,-0.551341,-1.132156,0.574711,False,False,False,False,True,False,True,True,False
4,0.051881,0.951331,-0.034755,-0.551341,-0.581981,-0.832432,True,False,True,False,True,False,False,False,True


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:,1]

In [15]:
svm = SVC(kernel="rbf", probability=True)
svm.fit(X_train, y_train)

y_pred_svm = svm.predict(X_test)
y_prob_svm = svm.predict_proba(X_test)[:,1]

In [19]:
# RANDOM FOREST
acc_rf = accuracy_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
roc_rf = roc_auc_score(y_test, y_prob_rf)

# SVM
acc_svm = accuracy_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
roc_svm = roc_auc_score(y_test, y_prob_svm)

results = pd.DataFrame({
    "Modelo": ["Random Forest", "SVM"],
    "Acurácia": [acc_rf, acc_svm],
    "Recall": [recall_rf, recall_svm],
    "ROC-AUC": [roc_rf, roc_svm]
})

results


,Modelo,Acurácia,Recall,ROC-AUC
0,Random Forest,0.864130,0.878505,0.936521
1,SVM,0.858696,0.869159,0.947202


In [20]:
print("CONCLUSÕES DA QUESTÃO 5")
print("---------------------------------------")
print("Melhor modelo:", "Random Forest" if roc_rf > roc_svm else "SVM")
print("\nMétricas:")
print(results)
print("\nVariáveis mais importantes:")
print(importances.head())


CONCLUSÕES DA QUESTÃO 5
---------------------------------------
Melhor modelo: SVM

Métricas:
          Modelo  Acurácia    Recall   ROC-AUC
0  Random Forest  0.864130  0.878505  0.936521
1            SVM  0.858696  0.869159  0.947202

Variáveis mais importantes:
ST_Slope_Up      0.148835
Oldpeak          0.117641
ST_Slope_Flat    0.112993
MaxHR            0.112760
Cholesterol      0.099510
dtype: float64
